# Boids Separation Influence Field

A JAX flock simulation where separation is driven by a sampled influence field.

In [1]:
import asyncio

import jax
import jax.numpy as jnp
import ipywidgets as widgets
from IPython.display import display
from ipycanvas import Canvas, hold_canvas
from jax.scipy.ndimage import map_coordinates

from flock_sim import (
    initialize_flock,
    separation_density,
    simulation_to_canvas,
    update_flock,
)

In [2]:
population = 200
simulation_grid_size = 128
render_grid_size = 600

dt = 0.016
min_velocity = 0.05
max_velocity = 1.0

separation_strength = 1.0
turn_rate = 0.2
sigma = 0.08

In [3]:
rng_key = jax.random.key(10)
flock = initialize_flock(rng_key, population)

update_step = jax.jit(
    lambda flock: update_flock(
        flock=flock,
        simulation_grid_size=simulation_grid_size,
        dt=dt,
        min_velocity=min_velocity,
        max_velocity=max_velocity,
        separation_strength=separation_strength,
        turn_rate=turn_rate,
        sigma=sigma,
    )
)

In [4]:
field_canvas = Canvas(width=render_grid_size, height=render_grid_size)
bird_canvas = Canvas(width=render_grid_size, height=render_grid_size)


def draw_density_normal_map(canvas, density):
    density = density / jnp.maximum(jnp.max(density), 1e-8)

    rows = jnp.linspace(0, simulation_grid_size - 1, render_grid_size)
    cols = jnp.linspace(0, simulation_grid_size - 1, render_grid_size)
    grid_rows, grid_cols = jnp.meshgrid(rows, cols, indexing="ij")
    height_map = map_coordinates(density, jnp.stack([grid_rows, grid_cols]), order=1)

    slope_y, slope_x = jnp.gradient(height_map)
    normal_strength = 20.0
    normal_z = jnp.full_like(height_map, 1.0 / normal_strength)
    normals = jnp.stack([-slope_x, -slope_y, normal_z], axis=-1)
    normals = normals / jnp.maximum(jnp.linalg.norm(normals, axis=-1, keepdims=True), 1e-8)

    rgb = ((normals * 0.5 + 0.5) * 255).astype(jnp.uint8)
    alpha = jnp.full_like(height_map, 255, dtype=jnp.uint8)
    image = jnp.concatenate([rgb, alpha[..., None]], axis=-1)
    canvas.put_image_data(jax.device_get(image), 0, 0)


def draw_birds(canvas, flock):
    points = simulation_to_canvas(flock.positions, render_grid_size)
    points = jax.device_get(points)
    canvas.clear()
    canvas.fill_style = "rgb(8, 10, 14)"
    canvas.fill_rect(0, 0, render_grid_size, render_grid_size)
    canvas.fill_style = "rgba(0, 0, 0, 0.8)"
    for x, y in points:
        canvas.fill_circle(float(x), float(y), 4.0)
    canvas.fill_style = "white"
    for x, y in points:
        canvas.fill_circle(float(x), float(y), 2.4)


def draw_frame(flock):
    density = separation_density(flock, simulation_grid_size, sigma, separation_strength)
    with hold_canvas(field_canvas):
        draw_density_normal_map(field_canvas, density)
    with hold_canvas(bird_canvas):
        draw_birds(bird_canvas, flock)

In [5]:
simulation_running = False
simulation_paused = False
simulation_task = None

toggle_button = widgets.Button(description="Start", button_style="success")
display(toggle_button)


def set_button_state(running, paused=False):
    if not running:
        toggle_button.description = "Start"
        toggle_button.button_style = "success"
    elif paused:
        toggle_button.description = "Resume"
        toggle_button.button_style = "info"
    else:
        toggle_button.description = "Pause"
        toggle_button.button_style = "warning"


async def run_simulation():
    global flock, simulation_running, simulation_paused
    simulation_running = True
    set_button_state(running=True, paused=simulation_paused)

    try:
        while simulation_running:
            if not simulation_paused:
                flock = update_step(flock)
                draw_frame(flock)
            await asyncio.sleep(dt)
    except asyncio.CancelledError:
        pass
    finally:
        simulation_running = False
        simulation_paused = False
        set_button_state(running=False)


def start_simulation():
    global simulation_running, simulation_paused, simulation_task
    if simulation_task is not None and not simulation_task.done():
        return
    simulation_running = True
    simulation_paused = False
    simulation_task = asyncio.create_task(run_simulation())
    set_button_state(running=True)


def pause_simulation():
    global simulation_paused
    if not simulation_running:
        return
    simulation_paused = not simulation_paused
    set_button_state(running=True, paused=simulation_paused)


def toggle_simulation(_=None):
    if simulation_running:
        pause_simulation()
    else:
        start_simulation()


def stop_simulation(_=None):
    global simulation_running, simulation_paused, simulation_task
    simulation_running = False
    simulation_paused = False
    if simulation_task is not None and not simulation_task.done():
        simulation_task.cancel()
    set_button_state(running=False)


toggle_button.on_click(toggle_simulation)
draw_frame(flock)
display(widgets.HBox([bird_canvas, field_canvas]))

Button(button_style='success', description='Start', style=ButtonStyle())